# CloudExam System — Full Architecture Notebook

```
User → Nginx LB → FastAPI (3 containers) → WebHDFS → HDFS (namenode + datanode)
                        ↓
                  PostgreSQL (shared DB)
```

This notebook walks through every layer of the system — why each piece exists, how it connects, and what actually happens under the hood.


## File Structure

```
cloudexam/
│
├── main.py                    ← FastAPI app, mounts all routes + UI
├── docker-compose.yml         ← spins up entire system: Hadoop + Postgres + 3 app instances + nginx
├── nginx.conf                 ← round-robin load balancer config
├── Dockerfile                 ← builds each app container
├── hdfs-site.xml              ← HDFS config: enables WebHDFS REST API
├── requirements.txt
├── secret.key                 ← JWT signing secret
│
├── app/
│   ├── core/
│   │   ├── config.py          ← settings (SECRET_KEY, HDFS_BASE_PATH)
│   │   └── auth.py            ← JWT create/verify, bcrypt password hashing
│   │
│   ├── db/
│   │   ├── session.py         ← PostgreSQL engine + get_db() dependency + startup retry
│   │   └── models.py          ← User, File, Chunk ORM tables
│   │
│   ├── routes/
│   │   ├── auth_routes.py     ← POST /register, POST /login
│   │   └── file_routes.py     ← upload, download, delete, list endpoints
│   │
│   ├── services/
│   │   ├── encryption.py      ← Fernet encrypt/decrypt
│   │   ├── chunking.py        ← split file into 200KB parts, merge them back
│   │   ├── hdfs_client.py     ← WebHDFS REST calls (PUT/GET/DELETE/MKDIRS)
│   │   └── file_service.py    ← orchestrates upload & download pipeline
│   │
│   └── ui/
│       └── pages.py           ← server-rendered HTML for login/register/teacher/student
│
└── storage/
    ├── tmp/                   ← raw file lands here right after browser upload
    ├── chunks/                ← encrypted chunks live here briefly during up/down
    └── merged/                ← merged .enc file lives here briefly during download
```

Everything in `storage/` is ephemeral — files are deleted immediately after use. All permanent data lives inside HDFS.


## Docker Setup — One Compose, Everything Inside

The entire system is declared in a single `docker-compose.yml`. No manual `docker run` commands.

```
docker compose up -d --build
```

That one command boots:
- `namenode` — HDFS NameNode
- `datanode` — HDFS DataNode
- `db` — PostgreSQL
- `app1`, `app2`, `app3` — three identical FastAPI replicas
- `nginx` — load balancer, the only container exposed to the host

All containers join the same Docker bridge network: `hadoop-net`.
Docker DNS resolves hostnames — so `app1` can call `http://namenode:9870` directly.

```yaml
networks:
  hadoop-net:
    driver: bridge
```

Why a managed network instead of an external one?
- `docker compose down` cleans it up automatically
- No stale network conflicts
- Container names become DNS hostnames reliably

### Volumes — persistent data

Three named volumes are declared:

```yaml
volumes:
  namenode_data:    ← HDFS namespace metadata
  datanode_data:    ← actual block data
  pg_data:          ← PostgreSQL tables
```

Without these, a `docker compose down` would wipe all uploaded files and all user accounts.

### Dockerfile

```dockerfile
FROM python:3.10
WORKDIR /app
COPY . .
RUN pip install --no-cache-dir -r requirements.txt
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

Simple and intentional — each container is a clean Python process, no state. Code is baked into the image at build time, so any code change requires `docker compose up --build`.


## Nginx — Round-Robin Load Balancer

```nginx
upstream fastapi_backend {
    server fastapi_app1:8000;
    server fastapi_app2:8000;
    server fastapi_app3:8000;
}

server {
    listen 80;
    location / {
        proxy_pass http://fastapi_backend;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
    }
}
```

Nginx distributes requests across the three app containers using round-robin — request 1 → app1, request 2 → app2, request 3 → app3, then back to app1.

Only nginx is exposed to the host on port 8000. The three app containers are never directly reachable from outside Docker.

```
Browser → localhost:8000 → nginx:80 → (app1 | app2 | app3):8000
```

The `/whoami` endpoint lets you verify Nginx is actually rotating:

```python
@app.get("/whoami")
def whoami():
    return {"container": socket.gethostname()}
```

```bash
for i in {1..6}; do curl http://localhost:8000/whoami; echo; done
# fastapi_app1
# fastapi_app2
# fastapi_app3
# fastapi_app1
# ...
```


## Database — PostgreSQL (Three Tables)

Earlier versions used SQLite. The problem: three FastAPI containers each got their own isolated copy of the database file. Registering on `app1` meant `app2` had no record of that user. PostgreSQL fixes this — one shared server, all three containers connect to it.

```yaml
db:
  image: postgres:15
  environment:
    POSTGRES_USER: cloud
    POSTGRES_PASSWORD: cloud
    POSTGRES_DB: cloudexam
  volumes:
    - pg_data:/var/lib/postgresql/data
  networks:
    - hadoop-net
```

### users

```
id | username | password_hash | role | encryption_key
```

Every user gets their own Fernet key generated at registration time. This key is stored in the DB and used server-side to encrypt/decrypt that user's files only.

### files

```
file_id | user_id | filename | visibility | status | hdfs_base_path
```

`visibility` is either `"teacher"` or `"student"` — controls who can see and access what.
`status` tracks the upload lifecycle: `UPLOADING` → `READY` (or `FAILED`).
`hdfs_base_path` is the HDFS directory for this file's chunks: `/cloud/{visibility}/{user_id}/{file_id}`

### chunks

```
chunk_id | file_id | chunk_index | hdfs_path
```

Every chunk of every file has its own row with the exact HDFS path. On download, chunks are fetched in `chunk_index` order and merged.

### DB Startup Retry

All three FastAPI containers start almost simultaneously and immediately try to connect to PostgreSQL. The DB container takes a few seconds to initialize, so connections fail at first. `depends_on` only guarantees the container started, not that it's ready.

```python
# session.py
for i in range(10):
    try:
        engine = create_engine(DATABASE_URL, pool_pre_ping=True)
        conn = engine.connect()
        conn.close()
        print("Connected to PostgreSQL")
        break
    except Exception:
        print(f"Waiting for DB... ({i+1}/10)")
        time.sleep(2)
```

The retry loop waits up to 20 seconds, checking every 2 seconds. This is a standard pattern in distributed startup — not a workaround.

### Table Creation

Tables are created automatically at startup using SQLAlchemy's `create_all`:

```python
@app.on_event("startup")
def startup():
    Base.metadata.create_all(bind=engine)
```

If the tables already exist, this is a no-op.


## Auth — Bcrypt + JWT

### Registration

```
POST /register  →  hash password (bcrypt)  +  generate Fernet key  →  save User to DB
```

```python
user = User(
    username=username,
    password_hash=hash_password(password),
    encryption_key=Fernet.generate_key().decode(),
    role=role
)
```

The Fernet key is a base64-encoded 32-byte random key. It never leaves the server — stored in PostgreSQL, used server-side for all encrypt/decrypt operations.

### Login

```
POST /login  →  verify password (bcrypt)  →  issue JWT with {user_id, role, exp}
```

```python
def create_token(user_id, role):
    payload = {
        "user_id": user_id,
        "role": role,
        "exp": datetime.utcnow() + timedelta(hours=6)
    }
    return jwt.encode(payload, settings.SECRET_KEY, algorithm="HS256")
```

The token is returned to the browser, stored in `localStorage`, and sent as `Authorization: Bearer <token>` on every subsequent request.

### Route Protection

```python
def get_current_user(token: str = Depends(oauth2)):
    payload = jwt.decode(token, settings.SECRET_KEY, algorithms=["HS256"])
    return payload   # → {user_id, role, exp}
```

Any route that does `Depends(get_current_user)` automatically rejects requests with missing or invalid tokens.


## WebHDFS — The HTTP Interface to HDFS

Files go into HDFS over pure HTTP using the WebHDFS REST API — no subprocess calls, no `docker exec`, no shell commands.

```python
NAMENODE = "http://namenode:9870/webhdfs/v1"
```

### Enabling WebHDFS (The Hidden Default)

Port 9870 is the NameNode UI — it always responds. But `/webhdfs/v1/...` on the same port is a separate REST API that is **off by default** in the Docker image.

```bash
docker exec namenode hdfs getconf -confKey dfs.webhdfs.enabled
# (no output = disabled, all requests return 403 or 404)
```

Fix: create `hdfs-site.xml` in the project root and mount it into the namenode container.

```xml
<configuration>
  <property>
    <name>dfs.namenode.name.dir</name>
    <value>file:///hadoop/dfs/name</value>
  </property>
  <property>
    <name>dfs.webhdfs.enabled</name>
    <value>true</value>
  </property>
</configuration>
```

```yaml
# docker-compose.yml
namenode:
  volumes:
    - namenode_data:/hadoop/dfs/name
    - ./hdfs-site.xml:/etc/hadoop/hdfs-site.xml
```

```bash
docker compose down
docker compose up --build
docker exec namenode hdfs getconf -confKey dfs.webhdfs.enabled
# → true
```

`dfs.namenode.name.dir` → where namenode stores its metadata on disk (persistent)
`dfs.webhdfs.enabled=true` → enables the REST API on the same port 9870

### Upload — Two-Step Redirect Protocol

WebHDFS upload is a two-step handshake because NameNode and DataNode are separate services:

```
Step 1:  PUT  /webhdfs/v1/cloud/.../chunk?op=CREATE
         → NameNode returns HTTP 307 with Location header → points to a DataNode URL

Step 2:  PUT  <DataNode URL>   with file data in body
         → DataNode returns 201 Created
```

```python
def upload(local_path, hdfs_path):
    init_url = f"{NAMENODE}{hdfs_path}?op=CREATE&overwrite=true&user.name=hadoop"

    r = requests.put(init_url, allow_redirects=False)  # must NOT auto-follow
    # r.status_code == 307
    upload_url = r.headers["Location"]

    # fix Docker hostname (see pitfalls below)
    upload_url = upload_url.replace("localhost", "datanode")

    with open(local_path, "rb") as f:
        r2 = requests.put(upload_url, data=f)
    # r2.status_code == 201
```

`allow_redirects=False` is critical — without it, `requests` follows the redirect automatically but loses the request body.

### Download

```python
def download(hdfs_path, local_path):
    url = f"{NAMENODE}{hdfs_path}?op=OPEN&user.name=hadoop"
    r = requests.get(url, stream=True)
    with open(local_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
```

`op=OPEN` streams the file from DataNode. `stream=True` + `iter_content` avoids loading the whole file into memory.

### Delete

```python
def delete_hdfs(hdfs_path):
    requests.delete(f"{NAMENODE}{hdfs_path}?op=DELETE&recursive=true&user.name=hadoop")
```

`recursive=true` deletes the entire directory tree for that file (all its chunks in one call).

### Directory Creation — Lazy MKDIRS

Before uploading chunks, the target HDFS directory must exist. Rather than running a manual `hdfs dfs -mkdir`, the upload pipeline calls `MKDIRS` via WebHDFS automatically:

```python
def ensure_hdfs_dir(hdfs_path):
    url = f"{NAMENODE}{hdfs_path}?op=MKDIRS&user.name=hadoop"
    r = requests.put(url)
    if r.status_code not in (200, 201):
        raise Exception(f"MKDIRS failed: {r.status_code} {r.text}")
```

This is called once per upload, right after the base path is computed. `MKDIRS` is idempotent — safe to call even if the directory already exists.

```python
base_path = f"/cloud/{visibility}/{user.id}/{file_id}"
ensure_hdfs_dir(base_path)   # ← creates the whole nested path
```


## WebHDFS Pitfalls in Docker

Two real issues show up when running WebHDFS inside Docker that don't exist in a normal Hadoop cluster.

### Pitfall 1 — Redirect to `localhost`

When the NameNode responds to an upload request with a 307 redirect, the `Location` header contains the DataNode's address:

```
Location: http://localhost:9864/webhdfs/v1/...
```

Hadoop assumes the client is external and uses `localhost` as the DataNode address. But inside Docker, `localhost` resolves to the FastAPI container itself — not the DataNode container.

Fix:

```python
upload_url = r.headers.get("Location")
upload_url = upload_url.replace("localhost", "datanode")  # ← use Docker DNS name
```

### Pitfall 2 — HDFS Permission Denied (user.name)

WebHDFS enforces Linux-style file permissions. Every operation needs a user identity. Without it, the NameNode treats the request as anonymous and blocks writes:

```json
{"RemoteException": {"exception": "AccessControlException",
  "message": "Permission denied: user=hadoop, access=WRITE,
              inode=\"/cloud\":root:supergroup:drwxr-xr-x"}}
```

Two things needed:

**1. Pass `user.name=hadoop` on every WebHDFS call**

```python
# every op param includes &user.name=hadoop
f"{NAMENODE}{path}?op=CREATE&overwrite=true&user.name=hadoop"
f"{NAMENODE}{path}?op=OPEN&user.name=hadoop"
f"{NAMENODE}{path}?op=MKDIRS&user.name=hadoop"
f"{NAMENODE}{path}?op=DELETE&recursive=true&user.name=hadoop"
```

This tells HDFS: execute this request as the `hadoop` OS user (no Kerberos, pseudo-authentication mode).

**2. Set correct ownership on `/cloud` after first boot**

The `/cloud` directory is created by `root` initially. The `hadoop` user can't write into a directory it doesn't own, even with `user.name=hadoop` on the request.

```bash
docker exec -it namenode hdfs dfs -chown -R hadoop:hadoop /cloud
docker exec -it namenode hdfs dfs -chmod -R 755 /cloud
```

- `chown` → makes `hadoop` the owner (identity matches the requests)
- `chmod 755` → owner can write, others can only read (controlled, not wide-open)

These run once after cluster first boot. Since HDFS data is persisted in named volumes, this survives restarts.


## Encryption — Fernet (Per-User Key)

Fernet is symmetric AES-128-CBC with HMAC-SHA256 from the `cryptography` library. Each user has their own key generated at registration time.

### Encrypt

```python
def encrypt(path, key):
    cipher = Fernet(key.encode())
    data = open(path, "rb").read()
    out = path + ".enc"
    open(out, "wb").write(cipher.encrypt(data))
    return out
```

Input: `exam.pdf` → Output: `exam.pdf.enc`

### Decrypt

```python
def decrypt(path, key):
    cipher = Fernet(key.encode())
    data = open(path, "rb").read()
    out = path.replace(".enc", "")
    open(out, "wb").write(cipher.decrypt(data))
    return out
```

Input: `<file_id>.enc` → Output: `<file_id>` (the original bytes)

Important design decision: on download, the system always uses the **file owner's** key, not the downloader's key. This is why `file.user.encryption_key` is used instead of `user.encryption_key` in `handle_download`. Teachers can download any student file — they need the student's key, not their own.


## Chunking — Split and Merge

Files are split into 200KB chunks before going into HDFS. This mirrors how real HDFS works (128MB blocks by default). Here it's done at the Python layer for explicit control.

### Split

```python
def split_file(path, chunk_size, out_dir):
    with open(path, "rb") as f:
        i = 0
        while True:
            data = f.read(chunk_size)
            if not data: break
            chunk_path = f"{out_dir}/{basename}_part{i}"
            open(chunk_path, "wb").write(data)
            i += 1
    return [list of chunk paths]
```

A 1MB file becomes 5 chunks: `file.enc_part0` through `file.enc_part4`.

### Merge

```python
def merge_chunks(paths, out_path):
    with open(out_path, "wb") as out:
        for p in paths:   # order matters — sorted by chunk_index from DB
            out.write(open(p, "rb").read())
    return out_path
```

On download, chunks are fetched from the DB ordered by `chunk_index` to guarantee correct merge order. Order isn't assumed from filenames — it's enforced at the DB query level.


## Upload Flow — Full Pipeline

```
Browser → POST /upload (multipart file)
    │
    ▼
file_routes.py → writes file to storage/tmp/<filename>
    │
    ▼
file_service.handle_upload()
    │
    ├─ ensure_hdfs_dir(base_path)   ← MKDIRS via WebHDFS
    ├─ create File row in DB (status=UPLOADING)
    ├─ encrypt(file_path, user.encryption_key)  → storage/tmp/<filename>.enc
    ├─ split_file(enc_file, 200KB, storage/chunks/)  → [chunk0, chunk1, ...]
    │
    └─ for each chunk:
        ├─ hdfs_client.upload(chunk_path, /cloud/{visibility}/{user_id}/{file_id}/{chunk_name})
        ├─ save Chunk row to DB with hdfs_path
        └─ os.remove(chunk_path)   ← delete local immediately after push
    │
    ├─ os.remove(enc_file)   ← delete encrypted source
    └─ File.status = READY
```

HDFS path structure:

```
/cloud/teacher/3/abc-uuid/exam.pdf.enc_part0
/cloud/teacher/3/abc-uuid/exam.pdf.enc_part1
/cloud/student/7/def-uuid/submission.pdf.enc_part0
```

State after upload:

```
storage/tmp/    → empty
storage/chunks/ → empty
HDFS            → has all chunks
DB              → has File row (READY) + one Chunk row per chunk
```


## Download Flow — Full Pipeline

```
Browser → GET /download?file_id=abc
    │
    ▼
file_routes.py → get current user from JWT
    │
    ▼
file_service.handle_download(file_id, user, db)
    │
    ├─ load File + its Chunks from DB (ordered by chunk_index)
    ├─ access control check:
    │    teacher → can download any file
    │    student → can only download their own student files
    │              (teacher files via /download-by-name only)
    │
    ├─ for each chunk:
    │    hdfs_client.download(chunk.hdfs_path, storage/chunks/<chunk_name>)
    │
    ├─ merge_chunks([chunk paths], storage/merged/<file_id>.enc)
    ├─ decrypt(merged_enc, file.user.encryption_key)  ← owner's key, not caller's
    │
    └─ return (final_path, filename, [all temp paths])
    │
    ▼
FileResponse(final_path, filename=filename)
    │
    ▼
BackgroundTask: delete all temp files after response is sent
```

State after download:

```
storage/chunks/ → empty
storage/merged/ → empty
HDFS            → unchanged (data still there)
```


## Access Control — Roles and Visibility

Two roles, two visibility values, two ways to download.

| role | uploads as | sees in UI | can download |
|------|-----------|------------|--------------|
| teacher | visibility="teacher" | all files via /all-files | any file by file_id |
| student | visibility="student" | own files via /my-files | own files by file_id; teacher files by filename |

Teacher uploads an exam paper → `visibility=teacher`, stored under `/cloud/teacher/...`
Student submits work → `visibility=student`, stored under `/cloud/student/...`

Students can't see other students' files. Students access teacher files using `/download-by-name?filename=exam.pdf` — the system finds the teacher-visible file by filename and serves it.

```python
# student downloading a teacher's file
file = db.query(File).filter_by(
    filename=filename,
    visibility="teacher"
).first()
```

On `/my-files`, students only see their own files because the query filters by `user_id`.


## Routes Overview

### Auth routes

```
POST /register   body: username, password, role
POST /login      body: username, password   →  returns JWT token
```

### File routes (all require `Authorization: Bearer <token>`)

```
POST   /upload              multipart file
GET    /download            ?file_id=<id>
GET    /download-by-name    ?filename=<name>   (student accessing teacher files)
DELETE /delete              ?file_id=<id>
GET    /my-files                               (student: own files only)
GET    /all-files                              (teacher only: all files)
```

### UI routes (HTML pages, no auth needed — JS handles token in localStorage)

```
GET /login
GET /register
GET /teacher
GET /student
GET /whoami     returns {"container": "fastapi_app2"}  (load balancing debug)
```


## Frontend — Server-Rendered HTML + Vanilla JS

The UI is generated server-side in `pages.py` using Python f-strings. No React, no templates — just raw HTML returned as `HTMLResponse`.

### Login flow (browser side)

```javascript
async function login(){
    let d = new FormData();
    d.append("username", username.value);
    d.append("password", password.value);

    let r = await fetch("/login", {method:"POST", body:d});
    let j = await r.json();

    localStorage.setItem("token", j.access_token);
    localStorage.setItem("role", j.role);

    location = j.role === "teacher" ? "/teacher" : "/student";
}
```

### safeFetch — the important wrapper

Every API call goes through this instead of raw `fetch()`:

```javascript
async function safeFetch(url, options={}){
    let token = localStorage.getItem("token");
    if(!token){ location="/login"; return null; }

    options.headers = options.headers || {};
    options.headers["Authorization"] = "Bearer " + token;

    let r = await fetch(url, options);

    if(r.status === 401){
        alert("Session expired. Please login again.");
        localStorage.clear();
        location="/login";
        return null;
    }
    return r;
}
```

This prevents silent failures — if the token expires or is missing, the user is redirected to login instead of seeing a cryptic error.

### File download in browser

```javascript
async function download(id, name){
    let r = await safeFetch("/download?file_id="+id);
    let b = await r.blob();

    let a = document.createElement("a");
    a.href = URL.createObjectURL(b);
    a.download = name;
    a.click();
}
```

The server returns a `FileResponse` which the browser receives as a binary blob, then triggers a programmatic `<a>` click to save it.


## HDFS Internals — Replication and the NameNode UI

With one DataNode (the current setup):

```
Block 0 → DataNode1 only
```

Even if replication factor is set to 3, HDFS can't physically replicate — there's only one place to put data. The cluster still works, you just lose fault tolerance.

With three DataNodes:

```
Block 0 → DN1, DN2, DN3
Block 1 → DN2, DN3, DN1
Block 2 → DN3, DN1, DN2
```

NameNode handles this automatically — it tracks block locations and assigns DataNodes.

### Browsing HDFS in the UI

Go to `http://localhost:9870` → Utilities → Browse Directory

You can see:
- File name
- Size
- Replication factor
- Block size
- Directory listing under `/cloud/`

This is how you verify files actually made it into HDFS after an upload.

The NameNode is the brain, DataNode is the storage. NameNode stores only metadata — which blocks exist, where they are, which DataNodes are alive. DataNode stores the actual bytes. When you do `hdfs dfs -ls /cloud` you're asking the NameNode. When data actually moves it goes directly to/from a DataNode.


## How to Run

### Step 1 — Start everything

```bash
docker compose up -d --build
```

### Step 2 — Verify WebHDFS is enabled

```bash
docker exec namenode hdfs getconf -confKey dfs.webhdfs.enabled
# → true
```

If not `true`, your `hdfs-site.xml` mount failed — check the path.

### Step 3 — Fix HDFS permissions (once after first boot)

```bash
docker exec -it namenode hdfs dfs -mkdir -p /cloud
docker exec -it namenode hdfs dfs -chown -R hadoop:hadoop /cloud
docker exec -it namenode hdfs dfs -chmod -R 755 /cloud
```

This only needs to be done once since HDFS data is in a named volume.

### Step 4 — Open the app

```
http://localhost:8000
```

You'll be redirected to `/login`.

### Step 5 — Test load balancing

```bash
for i in {1..6}; do curl http://localhost:8000/whoami; echo; done
# {"container": "fastapi_app1"}
# {"container": "fastapi_app2"}
# {"container": "fastapi_app3"}
# ...
```

### Step 6 — Inspect the DB

```bash
docker exec -it postgres_db psql -U cloud -d cloudexam
\dt
SELECT * FROM users;
SELECT * FROM files;
SELECT * FROM chunks;
```
